# libraries and Load Sentiment pipeline

In [1]:
!pip install transformers datasets scikit-learn torch -q
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

# Test 5 Sentance

In [2]:
sentences = [
    "I love learning Artificial Intelligence.",
    "This movie was terrible.",
    "The lecture was boring but informative.",
    "I am very happy with my results.",
    "The service was not good at all."
]

results = sentiment_pipeline(sentences)

for s, r in zip(sentences, results):
    print(f"Sentence: {s}")
    print(f"Label: {r['label']}")
    print(f"Confidence: {r['score']:.4f}")
    print("-"*50)

Sentence: I love learning Artificial Intelligence.
Label: POSITIVE
Confidence: 0.9995
--------------------------------------------------
Sentence: This movie was terrible.
Label: NEGATIVE
Confidence: 0.9997
--------------------------------------------------
Sentence: The lecture was boring but informative.
Label: POSITIVE
Confidence: 0.9352
--------------------------------------------------
Sentence: I am very happy with my results.
Label: POSITIVE
Confidence: 0.9999
--------------------------------------------------
Sentence: The service was not good at all.
Label: NEGATIVE
Confidence: 0.9998
--------------------------------------------------


# Load Tokenizer

In [3]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Tokanizer sample sentances

In [4]:
sample_sentences = [
    "Transformers are powerful models.",
    "Deep learning is amazing.",
    "BERT understands context."
]

encoded = tokenizer(
    sample_sentences,
    padding=True,
    truncation=True,
    max_length=10,
    return_tensors="pt"
)

print("Input IDs:\n", encoded["input_ids"])
print("\nAttention Mask:\n", encoded["attention_mask"])

Input IDs:
 tensor([[  101, 19081,  2024,  3928,  4275,  1012,   102],
        [  101,  2784,  4083,  2003,  6429,  1012,   102],
        [  101, 14324, 19821,  6123,  1012,   102,     0]])

Attention Mask:
 tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]])


# Load Dataset

In [7]:
from datasets import load_dataset

dataset = load_dataset("imdb")

# IMDb contains:
# 25,000 train
# 25,000 test
# Binary labels (0 = Negative, 1 = Positive

# Use Small Dataset

In [6]:
small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_test = dataset["test"].shuffle(seed=42).select(range(200))

# Tokenize Dataset

In [8]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# Load Pre-Train Model

In [9]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training Argument

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Trainer and maintainer

In [12]:
from transformers import Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# train model

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.4310075467225736, metrics={'train_runtime': 2128.5828, 'train_samples_per_second': 0.705, 'train_steps_per_second': 0.089, 'total_flos': 98666645760000.0, 'train_loss': 0.4310075467225736, 'epoch': 3.0})

# Evaluate Model

In [14]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.3947955071926117,
 'eval_accuracy': 0.825,
 'eval_f1': 0.8258706467661692,
 'eval_runtime': 129.6223,
 'eval_samples_per_second': 1.543,
 'eval_steps_per_second': 0.193,
 'epoch': 3.0}

In [15]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)
print(cm)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[[82 22]
 [13 83]]


## Why Transformers outperform RNNs?

Transformers process all words in parallel, unlike RNNs which process sequentially. This allows faster training and better handling of long-range dependencies. RNNs suffer from vanishing gradients, but Transformers use self-attention to capture global context efficiently. Therefore, Transformers scale better and achieve higher accuracy.


## What is Self-Attention?

Self-attention is a mechanism that allows each word in a sentence to focus on all other words. It assigns importance scores to capture relationships between words. This helps the model understand context, meaning, and dependencies within the sentence.